# BnSentMix — Model 4: Dual-Polarity XLM-R

## What Model 3 established

Model 3 **rejected H3**. Class-balanced focal loss produced mean validation macro-F1 `0.6878 ± 0.0098` and mixed-F1 `0.5593 ± 0.0308`, compared with Model 2's `0.7182 ± 0.0041` and `0.6282 ± 0.0162`. Every seed became worse.

For reference seed 42, mixed recall increased from `0.5464` to `0.6426`, but precision fell from `0.7195` to `0.5533`; mixed-F1 therefore fell from `0.6211` to `0.5946`. Model 3 predicted mixed for 338 validation examples versus 221 in Model 2, creating many more false positives. The result indicates that **class imbalance alone is not the main bottleneck**.

Model 4 now tests the study's central idea: mixed sentiment may be easier to learn as the simultaneous presence of positive and negative evidence than as one rare, flat class.

## Hypothesis H4 and literature basis

**H4:** Under the same duplicate-group-disjoint protocol, factorizing the four labels into independent positive- and negative-evidence targets will improve mixed-class F1 over flat four-class XLM-R, without reducing mean macro-F1 by more than `0.005`.

The [BnSentMix paper](https://aclanthology.org/2025.loreslm-1.4.pdf) defines mixed sentiment as positive and negative sentiments conveyed in different parts of the same text. Tan et al.'s [conflict-opinion model](https://aclanthology.org/D19-1342/) similarly reformulates positive/negative/neutral/conflict classification as two questions: whether positive evidence is present and whether negative evidence is present. Model 4 adapts that label factorization to BnSentMix, while leaving clause-aware modelling for Model 5.

| Original label | Positive evidence | Negative evidence | Decoded class |
|---:|---:|---:|---|
| 0 positive | 1 | 0 | positive |
| 1 negative | 0 | 1 | negative |
| 2 neutral | 0 | 0 | neutral |
| 3 mixed | 1 | 1 | mixed |

Two sigmoid outputs are trained with ordinary binary cross-entropy. Both thresholds are fixed at `0.5` before training; no class weighting, focal loss, resampling, or threshold tuning is used.

## Frozen protocol and decision rule

| Item | Fixed value |
|---|---|
| Dataset | `/kaggle/input/datasets/olidopu/bnsentmix/dataset.csv` |
| Split | Same verified group-disjoint 70/15/15 split |
| Backbone/head | `FacebookAI/xlm-roberta-base`; same classification-head structure |
| Model 4 change | 4-way softmax → 2 independent sigmoid evidence outputs |
| Loss | Unweighted binary cross-entropy |
| Evidence thresholds | Positive `0.5`; negative `0.5` |
| Maximum length | 128 tokens |
| Per-device batch / accumulation | 16 / 2 |
| Learning rate / weight decay | 2e-5 / 0.01 |
| Epoch budget / patience | 5 / 2 |
| Seeds | 13, 42, 77 |
| Checkpoint metric | Validation macro-F1 |
| Test policy | Sealed until Models 1–6 are frozen |

**Predeclared H4 support rule:** mean mixed-F1 must exceed Model 2's `0.6282`, and mean macro-F1 must be at least `0.7132` (`0.7182 − 0.005`). Results across three seeds are descriptive, not a significance claim.

> Model 2 showed a LayerNorm key warning during automatic checkpoint reload. Model 4 uses exact in-memory state saving and strict reload, as Model 3 did. Rerun Model 2 with the same safe checkpoint procedure before the final paper table.

In [ ]:
# Imports and every experimental choice are declared before training.
from pathlib import Path
import gc
import hashlib
import inspect
import json
import math
import re
import unicodedata

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display
from torch.utils.data import Dataset
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, f1_score,
    precision_recall_fscore_support
)
from sklearn.model_selection import StratifiedGroupKFold
from transformers import (
    AutoModelForSequenceClassification, AutoTokenizer,
    DataCollatorWithPadding, Trainer, TrainerCallback,
    TrainingArguments, set_seed
)

DATA_PATH = Path('/kaggle/input/datasets/olidopu/bnsentmix/dataset.csv')
PROTOCOL_DIR = Path('/kaggle/working/bnsentmix_protocol_2026')
MODEL_DIR = Path('/kaggle/working/bnsentmix_model4_dual_polarity')
PROTOCOL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_RAW_SHA256 = '148f23eb3dc40c1012a973efec920eaccc39700a74e5bcfb56806b0bf389029d'
EXPECTED_SPLIT_SHA256 = '7be81e52030967319d43ddc1e1feaf9817f9c3011a9b914265c34b5ce0ab8ec5'
EXPECTED_SPLIT_COUNTS = {'train': 13862, 'validation': 2975, 'test': 2973}
SPLIT_SEED = 2026

# Confirmed mapping; do not repeat label auditing.
LABELS = [0, 1, 2, 3]
LABEL_NAMES = {0: 'positive', 1: 'negative', 2: 'neutral', 3: 'mixed'}
MIXED_LABEL = 3
TARGET_MAP = {0: [1.0, 0.0], 1: [0.0, 1.0], 2: [0.0, 0.0], 3: [1.0, 1.0]}
EVIDENCE_NAMES = ['positive_evidence', 'negative_evidence']
EVIDENCE_THRESHOLD = 0.5

MODEL_NAME = 'FacebookAI/xlm-roberta-base'
MAX_LENGTH = 128
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
GRADIENT_ACCUMULATION_STEPS = 2
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_FRACTION = 0.10
MAX_EPOCHS = 5
EARLY_STOPPING_PATIENCE = 2
RUN_SEEDS = [13, 42, 77]
REFERENCE_SEED = 42
OPEN_FINAL_TEST = False

MODEL2_MEAN_MACRO_F1 = 0.7182
MODEL2_MEAN_MIXED_F1 = 0.6282
MACRO_NONINFERIORITY_MARGIN = 0.005

USE_FP16 = torch.cuda.is_available()
GPU_COUNT = torch.cuda.device_count() if torch.cuda.is_available() else 0
print('GPU count:', GPU_COUNT)
print('Device 0:', torch.cuda.get_device_name(0) if GPU_COUNT else 'CPU')
print('Evidence threshold:', EVIDENCE_THRESHOLD)
print('Final test open:', OPEN_FINAL_TEST)

In [ ]:
# Reconstruct and verify the identical Model 1–3 split.
def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def normalized_group_key(text):
    text = unicodedata.normalize('NFKC', str(text)).lower()
    text = re.sub(r'[\u200b-\u200d\ufeff]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return hashlib.sha256(text.encode('utf-8')).hexdigest()

raw = pd.read_csv(DATA_PATH)
assert sha256_file(DATA_PATH) == EXPECTED_RAW_SHA256, 'Raw dataset differs from Models 1–3.'
lookup = {column.strip().lower(): column for column in raw.columns}
text_col = next(lookup[name] for name in ['sentence', 'text', 'comment'] if name in lookup)
label_col = next(lookup[name] for name in ['label', 'sentiment', 'class'] if name in lookup)
df = raw[[text_col, label_col]].rename(columns={text_col: 'text', label_col: 'label'}).copy()
df.insert(0, 'source_row_id', np.arange(len(df)))
df = df.dropna(subset=['text', 'label'])
df['text'] = df['text'].astype(str).str.strip()
df['label'] = df['label'].astype(int)
df = df[df['text'].ne('')].copy()
df['group_id'] = df['text'].map(normalized_group_key)
df = df.drop_duplicates(['group_id', 'label'], keep='first').reset_index(drop=True)

splitter = StratifiedGroupKFold(n_splits=20, shuffle=True, random_state=SPLIT_SEED)
fold_ids = np.full(len(df), -1, dtype=int)
for fold_id, (_, held_out) in enumerate(
    splitter.split(df['text'], df['label'], groups=df['group_id'])
):
    fold_ids[held_out] = fold_id
df['split'] = np.where(fold_ids < 14, 'train', np.where(fold_ids < 17, 'validation', 'test'))

assert df.groupby('group_id')['split'].nunique().max() == 1
assert df['split'].value_counts().to_dict() == EXPECTED_SPLIT_COUNTS
split_path = PROTOCOL_DIR / 'bnsentmix_locked_split.csv'
df.to_csv(split_path, index=False)
assert sha256_file(split_path) == EXPECTED_SPLIT_SHA256, 'Split differs from Models 1–3.'

train_df = df[df['split'].eq('train')].copy()
val_df = df[df['split'].eq('validation')].copy()
test_df = df[df['split'].eq('test')].copy()
display(pd.crosstab(df['split'], df['label']).rename(columns=LABEL_NAMES))
print('Verified split SHA-256:', EXPECTED_SPLIT_SHA256)

In [ ]:
# Convert the original four classes into two evidence targets.
def labels_to_evidence(labels):
    return np.asarray([TARGET_MAP[int(label)] for label in labels], dtype=np.float32)

def evidence_to_labels(evidence):
    evidence = np.asarray(evidence).astype(int)
    positive_on = evidence[:, 0] == 1
    negative_on = evidence[:, 1] == 1
    labels = np.full(len(evidence), 2, dtype=int)  # default: neutral [0, 0]
    labels[positive_on & ~negative_on] = 0
    labels[~positive_on & negative_on] = 1
    labels[positive_on & negative_on] = 3
    return labels

mapping_table = pd.DataFrame([
    {'label': label, 'class_name': LABEL_NAMES[label],
     'positive_target': TARGET_MAP[label][0], 'negative_target': TARGET_MAP[label][1]}
    for label in LABELS
])
train_evidence = labels_to_evidence(train_df['label'])
evidence_prevalence = pd.DataFrame({
    'evidence': EVIDENCE_NAMES,
    'positive_examples': train_evidence.sum(axis=0).astype(int),
    'training_prevalence': train_evidence.mean(axis=0),
})
display(mapping_table)
display(evidence_prevalence.round(4))
assert np.array_equal(evidence_to_labels(train_evidence), train_df['label'].to_numpy())

In [ ]:
# Tokenization and data handling remain the same except for two-value labels.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

class DualPolarityDataset(Dataset):
    def __init__(self, frame):
        self.texts = frame['text'].tolist()
        self.targets = labels_to_evidence(frame['label']).tolist()

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, index):
        encoded = tokenizer(
            self.texts[index], truncation=True, max_length=MAX_LENGTH
        )
        encoded['labels'] = self.targets[index]
        return encoded

train_dataset = DualPolarityDataset(train_df)
val_dataset = DualPolarityDataset(val_df)
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer, pad_to_multiple_of=8 if USE_FP16 else None
)

VISIBLE_DEVICES = max(GPU_COUNT, 1)
GLOBAL_EFFECTIVE_BATCH_SIZE = (
    TRAIN_BATCH_SIZE * VISIBLE_DEVICES * GRADIENT_ACCUMULATION_STEPS
)
BATCHES_PER_EPOCH = math.ceil(len(train_dataset) / (TRAIN_BATCH_SIZE * VISIBLE_DEVICES))
UPDATES_PER_EPOCH = math.ceil(BATCHES_PER_EPOCH / GRADIENT_ACCUMULATION_STEPS)
TOTAL_UPDATE_STEPS = UPDATES_PER_EPOCH * MAX_EPOCHS
WARMUP_STEPS = math.ceil(TOTAL_UPDATE_STEPS * WARMUP_FRACTION)
print('Training / validation:', len(train_dataset), '/', len(val_dataset))
print('Detected global effective batch size:', GLOBAL_EFFECTIVE_BATCH_SIZE)
print('Maximum optimizer steps / warmup steps:', TOTAL_UPDATE_STEPS, '/', WARMUP_STEPS)

In [ ]:
# Decode two evidence probabilities into the original four classes.
def sigmoid_numpy(logits):
    logits = np.clip(np.asarray(logits), -50, 50)
    return 1.0 / (1.0 + np.exp(-logits))

def predictions_from_logits(logits):
    probabilities = sigmoid_numpy(logits)
    evidence_predictions = (probabilities >= EVIDENCE_THRESHOLD).astype(int)
    class_predictions = evidence_to_labels(evidence_predictions)
    return probabilities, evidence_predictions, class_predictions

def calculate_metrics(true_evidence, predicted_evidence):
    true_evidence = np.asarray(true_evidence).astype(int)
    predicted_evidence = np.asarray(predicted_evidence).astype(int)
    labels = evidence_to_labels(true_evidence)
    predictions = evidence_to_labels(predicted_evidence)
    mixed_precision, mixed_recall, mixed_f1, _ = precision_recall_fscore_support(
        labels == MIXED_LABEL, predictions == MIXED_LABEL,
        average='binary', zero_division=0
    )
    return {
        'accuracy': float(accuracy_score(labels, predictions)),
        'macro_f1': float(f1_score(
            labels, predictions, labels=LABELS, average='macro', zero_division=0
        )),
        'weighted_f1': float(f1_score(
            labels, predictions, labels=LABELS, average='weighted', zero_division=0
        )),
        'mixed_precision': float(mixed_precision),
        'mixed_recall': float(mixed_recall),
        'mixed_f1': float(mixed_f1),
        'positive_evidence_f1': float(f1_score(
            true_evidence[:, 0], predicted_evidence[:, 0], zero_division=0
        )),
        'negative_evidence_f1': float(f1_score(
            true_evidence[:, 1], predicted_evidence[:, 1], zero_division=0
        )),
        'evidence_exact_match': float(np.mean(np.all(
            true_evidence == predicted_evidence, axis=1
        ))),
    }

def compute_metrics(eval_prediction):
    logits, true_evidence = eval_prediction
    if isinstance(logits, tuple):
        logits = logits[0]
    _, predicted_evidence, _ = predictions_from_logits(logits)
    return calculate_metrics(true_evidence, predicted_evidence)

In [ ]:
# Ordinary unweighted BCE isolates label factorization from Model 3's weighting.
class DualPolarityTrainer(Trainer):
    def compute_loss(
        self, model, inputs, return_outputs=False, num_items_in_batch=None
    ):
        targets = inputs.pop('labels').float()
        outputs = model(**inputs)
        loss = F.binary_cross_entropy_with_logits(outputs.logits, targets)
        return (loss, outputs) if return_outputs else loss

# Save the best live state directly to avoid checkpoint key conversion issues.
class ExactBestStateCallback(TrainerCallback):
    def __init__(self, state_path, patience):
        self.state_path = Path(state_path)
        self.patience = int(patience)
        self.best_score = -np.inf
        self.best_epoch = None
        self.best_loss = None
        self.bad_epochs = 0

    def on_evaluate(self, args, state, control, metrics=None, model=None, **kwargs):
        score = metrics.get('eval_macro_f1') if metrics else None
        if score is None:
            return control
        if score > self.best_score:
            self.best_score = float(score)
            self.best_epoch = float(state.epoch)
            self.best_loss = float(metrics['eval_loss'])
            self.bad_epochs = 0
            unwrapped_model = model.module if hasattr(model, 'module') else model
            cpu_state = {
                key: value.detach().cpu()
                for key, value in unwrapped_model.state_dict().items()
            }
            torch.save({'model_state_dict': cpu_state}, self.state_path)
        else:
            self.bad_epochs += 1
            if self.bad_epochs >= self.patience:
                control.should_training_stop = True
        return control

def load_exact_state(path):
    load_kwargs = {'map_location': 'cpu'}
    if 'weights_only' in inspect.signature(torch.load).parameters:
        load_kwargs['weights_only'] = True
    return torch.load(path, **load_kwargs)['model_state_dict']

In [ ]:
# Training schedule is identical to Model 3.
def make_training_arguments(seed, output_dir):
    arguments = dict(
        output_dir=str(output_dir),
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        num_train_epochs=MAX_EPOCHS,
        weight_decay=WEIGHT_DECAY,
        warmup_steps=WARMUP_STEPS,
        lr_scheduler_type='linear',
        max_grad_norm=1.0,
        save_strategy='no',
        logging_strategy='epoch',
        load_best_model_at_end=False,
        seed=seed,
        data_seed=seed,
        fp16=USE_FP16,
        dataloader_num_workers=2,
        report_to=[],
    )
    argument_names = inspect.signature(TrainingArguments.__init__).parameters
    arguments['eval_strategy' if 'eval_strategy' in argument_names else 'evaluation_strategy'] = 'epoch'
    return TrainingArguments(**arguments)

def build_fresh_model():
    # The usual XLM-R classification head is retained, but it now emits
    # two independent evidence logits instead of four softmax logits.
    return AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=2,
        id2label={0: EVIDENCE_NAMES[0], 1: EVIDENCE_NAMES[1]},
        label2id={EVIDENCE_NAMES[0]: 0, EVIDENCE_NAMES[1]: 1},
        problem_type='multi_label_classification',
    )

In [ ]:
# Train three independent seeds and retain validation-selected states.
def train_one_seed(seed):
    seed_dir = MODEL_DIR / f'seed_{seed}'
    seed_dir.mkdir(parents=True, exist_ok=True)
    best_state_path = seed_dir / 'best_state.pt'
    set_seed(seed)

    model = build_fresh_model()
    callback = ExactBestStateCallback(best_state_path, EARLY_STOPPING_PATIENCE)
    trainer = DualPolarityTrainer(
        model=model,
        args=make_training_arguments(seed, seed_dir),
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[callback],
    )
    trainer.train()

    if not best_state_path.exists():
        raise RuntimeError('No validation-selected state was saved.')
    model.load_state_dict(load_exact_state(best_state_path), strict=True)
    output = trainer.predict(val_dataset)
    logits = output.predictions[0] if isinstance(output.predictions, tuple) else output.predictions
    probabilities, evidence_predictions, class_predictions = predictions_from_logits(logits)
    true_evidence = labels_to_evidence(val_df['label'])
    metrics = calculate_metrics(true_evidence, evidence_predictions)
    metrics.update({
        'seed': seed,
        'best_epoch': callback.best_epoch,
        'validation_loss': callback.best_loss,
        'best_state_path': str(best_state_path),
    })

    prediction_frame = val_df[['source_row_id', 'group_id', 'label']].copy()
    prediction_frame['positive_probability'] = probabilities[:, 0]
    prediction_frame['negative_probability'] = probabilities[:, 1]
    prediction_frame['positive_evidence_prediction'] = evidence_predictions[:, 0]
    prediction_frame['negative_evidence_prediction'] = evidence_predictions[:, 1]
    prediction_frame['prediction'] = class_predictions
    prediction_frame.to_csv(seed_dir / 'validation_predictions.csv', index=False)
    pd.DataFrame(trainer.state.log_history).to_csv(
        seed_dir / 'training_log.csv', index=False
    )

    del trainer, model, output
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return metrics, probabilities, evidence_predictions, class_predictions

validation_rows = []
validation_outputs = {}
for run_seed in RUN_SEEDS:
    print(f'\n===== Training seed {run_seed} =====')
    metrics, probabilities, evidence_predictions, class_predictions = train_one_seed(run_seed)
    validation_rows.append(metrics)
    validation_outputs[run_seed] = {
        'probabilities': probabilities,
        'evidence_predictions': evidence_predictions,
        'class_predictions': class_predictions,
    }

validation_results = pd.DataFrame(validation_rows).sort_values('seed').reset_index(drop=True)
metric_columns = [
    'accuracy', 'macro_f1', 'weighted_f1',
    'mixed_precision', 'mixed_recall', 'mixed_f1',
    'positive_evidence_f1', 'negative_evidence_f1', 'evidence_exact_match'
]
validation_summary = validation_results[metric_columns].agg(['mean', 'std']).T
validation_results.to_csv(MODEL_DIR / 'validation_seed_results.csv', index=False)
validation_summary.to_csv(MODEL_DIR / 'validation_mean_std.csv')
tokenizer.save_pretrained(MODEL_DIR / 'tokenizer')
display(validation_results.drop(columns='best_state_path').round(4))
display(validation_summary.round(4))

In [ ]:
# Same-seed comparison against Models 2 and 3.
prior_results = pd.DataFrame({
    'seed': [13, 42, 77],
    'model2_macro_f1': [0.7158, 0.7159, 0.7229],
    'model2_mixed_f1': [0.6168, 0.6211, 0.6467],
    'model3_macro_f1': [0.6767, 0.6950, 0.6917],
    'model3_mixed_f1': [0.5383, 0.5946, 0.5449],
})
paired = prior_results.merge(validation_results, on='seed', validate='one_to_one')
paired['delta_macro_vs_model2'] = paired['macro_f1'] - paired['model2_macro_f1']
paired['delta_mixed_vs_model2'] = paired['mixed_f1'] - paired['model2_mixed_f1']
paired['delta_macro_vs_model3'] = paired['macro_f1'] - paired['model3_macro_f1']
paired['delta_mixed_vs_model3'] = paired['mixed_f1'] - paired['model3_mixed_f1']
comparison_columns = [
    'seed', 'model2_macro_f1', 'model3_macro_f1', 'macro_f1',
    'delta_macro_vs_model2', 'model2_mixed_f1', 'model3_mixed_f1',
    'mixed_f1', 'delta_mixed_vs_model2'
]
display(paired[comparison_columns].round(4))
paired.to_csv(MODEL_DIR / 'model2_model3_model4_paired_validation.csv', index=False)

mean_macro = float(validation_results['macro_f1'].mean())
mean_mixed = float(validation_results['mixed_f1'].mean())
delta_macro = mean_macro - MODEL2_MEAN_MACRO_F1
delta_mixed = mean_mixed - MODEL2_MEAN_MIXED_F1
h4_supported = (
    delta_mixed > 0 and
    delta_macro >= -MACRO_NONINFERIORITY_MARGIN
)
decision = pd.DataFrame([{
    'model4_mean_macro_f1': mean_macro,
    'delta_macro_vs_model2': delta_macro,
    'model4_mean_mixed_f1': mean_mixed,
    'delta_mixed_vs_model2': delta_mixed,
    'macro_margin': MACRO_NONINFERIORITY_MARGIN,
    'H4_supported_on_validation': h4_supported,
}])
decision.to_csv(MODEL_DIR / 'hypothesis_H4_decision.csv', index=False)
display(decision.round(4))
print(
    'Interpretation:',
    'H4 supported on validation.' if h4_supported else
    'H4 not supported under the predeclared validation rule.'
)

In [ ]:
# Mechanism diagnostics for the predeclared reference seed.
reference = validation_outputs[REFERENCE_SEED]
reference_predictions = reference['class_predictions']
reference_probabilities = reference['probabilities']
reference_evidence = reference['evidence_predictions']

reference_report = classification_report(
    val_df['label'], reference_predictions, labels=LABELS,
    target_names=[LABEL_NAMES[label] for label in LABELS],
    output_dict=True, zero_division=0
)
display(pd.DataFrame(reference_report).T.round(4))
display(pd.DataFrame(
    confusion_matrix(val_df['label'], reference_predictions, labels=LABELS),
    index=[f'true_{LABEL_NAMES[label]}' for label in LABELS],
    columns=[f'pred_{LABEL_NAMES[label]}' for label in LABELS],
))

# This table tests whether true mixed samples activate both evidence outputs.
diagnostic_rows = []
true_labels = val_df['label'].to_numpy()
for label in LABELS:
    mask = true_labels == label
    diagnostic_rows.append({
        'true_class': LABEL_NAMES[label],
        'support': int(mask.sum()),
        'mean_positive_probability': float(reference_probabilities[mask, 0].mean()),
        'mean_negative_probability': float(reference_probabilities[mask, 1].mean()),
        'positive_head_on_rate': float(reference_evidence[mask, 0].mean()),
        'negative_head_on_rate': float(reference_evidence[mask, 1].mean()),
        'both_heads_on_rate': float(np.mean(np.all(reference_evidence[mask] == 1, axis=1))),
    })
mechanism_table = pd.DataFrame(diagnostic_rows)
mechanism_table.to_csv(MODEL_DIR / 'reference_seed_evidence_diagnostics.csv', index=False)
display(mechanism_table.round(4))

In [ ]:
# Save the complete frozen experiment specification.
manifest = {
    'experiment': 'Model 4: dual-polarity XLM-R',
    'hypothesis': 'Explicit positive/negative evidence improves mixed detection',
    'dataset_sha256': EXPECTED_RAW_SHA256,
    'split_sha256': EXPECTED_SPLIT_SHA256,
    'model_name': MODEL_NAME,
    'target_map': TARGET_MAP,
    'evidence_threshold': EVIDENCE_THRESHOLD,
    'loss': 'unweighted binary cross entropy',
    'run_seeds': RUN_SEEDS,
    'global_effective_batch_size': GLOBAL_EFFECTIVE_BATCH_SIZE,
    'test_opened': OPEN_FINAL_TEST,
}
with open(MODEL_DIR / 'experiment_manifest.json', 'w') as handle:
    json.dump(manifest, handle, indent=2)

## Final-test gate

Keep `OPEN_FINAL_TEST=False`. Open the test partition only after Models 1–6, every hypothesis, hyperparameter, threshold, and selection rule are frozen. If H4 fails, retain the negative result and proceed to Model 5's clause/contrast-aware evidence pooling; do not tune this model post hoc.

In [ ]:
# Final evaluation is deliberately unreachable during development.
if not OPEN_FINAL_TEST:
    print('Final test remains sealed.')
else:
    test_dataset = DualPolarityDataset(test_df)
    true_test_evidence = labels_to_evidence(test_df['label'])
    test_rows = []

    for run_seed in RUN_SEEDS:
        state_path = MODEL_DIR / f'seed_{run_seed}' / 'best_state.pt'
        if not state_path.exists():
            raise FileNotFoundError(f'No selected state for seed {run_seed}.')
        model = build_fresh_model()
        model.load_state_dict(load_exact_state(state_path), strict=True)
        inference_args = TrainingArguments(
            output_dir=str(MODEL_DIR / f'test_seed_{run_seed}'),
            per_device_eval_batch_size=EVAL_BATCH_SIZE,
            fp16=USE_FP16, dataloader_num_workers=2, report_to=[]
        )
        inference_trainer = Trainer(
            model=model, args=inference_args, data_collator=data_collator
        )
        output = inference_trainer.predict(test_dataset)
        logits = output.predictions[0] if isinstance(output.predictions, tuple) else output.predictions
        probabilities, evidence_predictions, class_predictions = predictions_from_logits(logits)
        metrics = calculate_metrics(true_test_evidence, evidence_predictions)
        test_rows.append({'seed': run_seed, **metrics})

        prediction_frame = test_df[['source_row_id', 'group_id', 'label']].copy()
        prediction_frame['positive_probability'] = probabilities[:, 0]
        prediction_frame['negative_probability'] = probabilities[:, 1]
        prediction_frame['prediction'] = class_predictions
        prediction_frame.to_csv(
            MODEL_DIR / f'FINAL_test_predictions_seed_{run_seed}.csv', index=False
        )
        del inference_trainer, model, output
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    final_results = pd.DataFrame(test_rows)
    final_summary = final_results[metric_columns].agg(['mean', 'std']).T
    final_results.to_csv(MODEL_DIR / 'FINAL_test_seed_results.csv', index=False)
    final_summary.to_csv(MODEL_DIR / 'FINAL_test_mean_std.csv')
    display(final_results.round(4))
    display(final_summary.round(4))